<a href="https://colab.research.google.com/github/UDAYNANNAKA/rag-/blob/main/rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install pymupdf sentence-transformers scikit-learn google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 9.2 MB/s eta 0:00:00


In [ ]:
from google.colab import files
uploaded = files.upload()
pdf_path = next(iter(uploaded))
print("Uploaded:", pdf_path)

Saving PROGRAM RAG.docx - Google Docs.pdf to PROGRAM RAG.docx - Google Docs.pdf
Uploaded: PROGRAM RAG.docx - Google Docs.pdf


In [ ]:
import pymupdf
pdf = pymupdf.open(pdf_path)
pages = []
for page_number, page in enumerate(pdf, start=1):
  text = page.get_text("text")
  if text.strip():
    pages.append({"page": page_number, "text": text})
pdf.close()
print("Pages with text:", len(pages))

Pages with text: 6


In [ ]:
def chunk_text(text, chunk_size=800, overlap=120):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

chunks = []
for page in pages:
    for chunk in chunk_text(page["text"]):
        chunks.append({
            "page": page["page"],
            "text": chunk
        })
print("Total chunks:", len(chunks))

Total chunks: 15


In [ ]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
chunk_texts = [c["text"] for c in chunks]
chunk_embeddings = embedding_model.encode(chunk_texts,
show_progress_bar=False)
print("Embedding shape:", chunk_embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding shape: (15, 384)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
def retrieve(query, top_k=4):
  query_embedding = embedding_model.encode([query])
  scores = cosine_similarity(query_embedding, chunk_embeddings)[0]
  top_indices = scores.argsort()[-top_k:][::-1]
  results = []
  for i in top_indices:
    results.append({
      "page": chunks[i]["page"],
      "text": chunks[i]["text"],
      "score": float(scores[i])
    })
  return results

In [ ]:
query = "What is the main purpose of this document?"
results = retrieve(query, top_k=4)
for rank, r in enumerate(results, start=1):
  print(f"\nRank {rank} | Page {r['page']} | Score {r['score']:.4f}")
  print(r["text"][:500])


Rank 1 | Page 6 | Score 0.2700
eddings rather than a​
​persistent vector database, simple character-based chunking rather than structure-aware chunking, and no​
​reranker or formal RAG evaluation. Scanned/image-only PDFs may require OCR; PyMuPDF's documentation​
​explicitly notes OCR for image-based text.  citeturn0search0​
​17. What to Learn Next​
​PDF RAG → Vector Database (FAISS/Chroma) → Better Chunking → Metadata → Reranking → RAG Evaluation​
​→ Advanced RAG → Agentic AI → MCP​
​Official References​
​PyMuPDF text extra

Rank 2 | Page 3 | Score 0.2232

​results.append({​
​"page": chunks[i]["page"],​
​"text": chunks[i]["text"],​
​"score": float(scores[i])​
​})​
​return results​
​Explanation: The question is embedded, compared with every chunk, and the highest-scoring chunks are returned.​
​Cosine similarity is a standard similarity option in Sentence Transformers.  citeturn1search0turn1search4​
​8. Test Retrieval Before Using an LLM​
​Colab Cell 7​
​query = "What is the main p

In [ ]:
def build_prompt(query, results):
    context = "\n\n".join(
        f"[Page {r['page']}]\n{r['text']}"
        for r in results
    )

    return f"""
You are a PDF question-answering assistant.
Answer the user's question using ONLY the supplied context.
If the context does not contain the answer, say: 'The answer is not available in the provided PDF.'
Do not invent facts.
Mention the relevant PDF page number(s) when possible.

CONTEXT:
{context}

QUESTION:
{query}
ANSWER:
"""

In [ ]:
!pip -q install -U google-genai

from google import genai
from getpass import getpass

api_key = getpass("Enter your Gemini API key: ")
client = genai.Client(api_key=api_key)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.0/268.0 kB 19.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.61.0 which is incompatible.
Enter your Gemini API key: ··········


In [ ]:
query = "What are the steps to build a pdf chatbot?"
results = retrieve(query, top_k=4)
prompt = build_prompt(query, results)

response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents=prompt
)

print(response.text)


Based on the provided PDF, the process and steps to build the PDF chatbot include:

**The Complete Flow [Page 1]:**
1. **PDF**
2. **Text Extraction**
3. **Chunks** (splitting text into retrievable passages)
4. **Embeddings** (creating vectors)
5. **Similarity Search**
6. **Top-k Context**
7. **Prompt** (combining the question and retrieved context)
8. **LLM**
9. **Answer**

**Step-by-step tutorial sections outlined in the text:**
* **Step 2:** Install the Libraries (`pymupdf`, `sentence-transformers`, `scikit-learn`, `google-genai`) [Page 1]
* **Step 3:** Upload Your PDF [Page 1]
* **Step 4:** Extract Text from the PDF [Page 1]
* *(Intermediate steps 5–10 are omitted in the provided context)*
* **Step 11:** Generate the RAG Answer (retrieve relevant passages, build the prompt, and generate content with the LLM) [Page 5]
* **Step 12:** Turn It into a Chatbot (using a loop to repeatedly accept user questions, retrieve context, and generate answers until the user exits) [Page 5]


In [ ]:
while True:
    query = input("\nAsk a question about the PDF (type 'exit' to stop): ")

    if query.lower() == "exit":
        print("Chatbot stopped.")
        break

    results = retrieve(query, top_k=4)
    prompt = build_prompt(query, results)

    response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents=prompt
    )

    print("\nAnswer:\n", response.text)



Ask a question about the PDF (type 'exit' to stop): what are steps to build a chat bot

Answer:
 Based on the provided context, the complete flow and steps outlined to build the PDF chatbot are:

**Overall Flow:**
PDF → Text Extraction → Chunks → Embeddings → Similarity Search → Top-k Context → Prompt → LLM → Answer (Page 1). Specifically, it extracts the PDF text, splits it into chunks, creates embeddings, retrieves the most relevant chunks for a question, and gives only those chunks to the LLM (Page 1).

**Numbered Steps detailed in the document:**
1. **What You Will Build** [Page 1]
2. **Install the Libraries** (such as PyMuPDF, Sentence Transformers, scikit-learn, and google-genai) [Page 1]
3. **Upload Your PDF** [Page 1]
4. **Extract Text from the PDF** [Page 1]
*(Note: Steps 5 through 10 are missing from the provided context)*
11. **Generate the RAG Answer** (retrieve relevant passages, build a prompt, and generate content with the LLM) [Page 5]
12. **Turn It into a Chatbot** (c